# Inference - CollabLLM Finetuned Model

In [1]:
# %env OPENAI_API_KEY=
# %env ANTHROPIC_API_KEY=
# Or set these environment variables in your system
from dotenv import load_dotenv
YOUR_DOTENV_PATH = "../.env"
load_dotenv(YOUR_DOTENV_PATH)

# Disable logging for the collabllm package
# Set to 1 to see the process of the reward computation.
%env ENABLE_COLLABLLM_LOGGING=0 

env: ENABLE_COLLABLLM_LOGGING=0


## Example 1: Movie Recommendation

In [2]:
import sys
sys.path.append('..')

import logging
logging.getLogger("LiteLLM").setLevel(logging.CRITICAL)

In [3]:
task_desc = "Recommend a movie."
single_turn_prompt = "Find a film that suitable for a date night. It should deliver an epic romantic drama, ideally in the 20th-century America, and carry the same decades-long, nostalgic storytelling spirit as Forrest Gump."

# base_model_name = "meta-llama/Llama-3.2-3B-Instruct"
lora_adapter_path = "../checkpoints/llama-3b-checkpoint"

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig

# Load LoRA adapter configuration
print(f"Loading LoRA adapter from: {lora_adapter_path}")
peft_config = PeftConfig.from_pretrained(lora_adapter_path)
print(f"Base model for LoRA: {peft_config.base_model_name_or_path}")

# Use the actual base model from the LoRA config
actual_base_model = peft_config.base_model_name_or_path
print(f"Using base model from adapter config: {actual_base_model}")

# Load base model and tokenizer using the correct base model
print(f"Loading base model: {actual_base_model}")
tokenizer = AutoTokenizer.from_pretrained(actual_base_model, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    actual_base_model, 
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# Load LoRA adapter onto base model
print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, lora_adapter_path)
print("LoRA adapter loaded successfully!")

# Print model information
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded successfully!")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable percentage: {trainable_params/total_params:.2%}")

# Set up tokenizer padding
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def generate_response(messages, max_new_tokens=512, temperature=0.8):
    """Generate response from the model for given messages
    
    Args:
        messages: List of message dicts with 'role' and 'content' keys
                 OR a single string (will be treated as user message)
    """
    # Handle backward compatibility with string input
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    
    # Apply chat template
    formatted_prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    # Tokenize
    inputs = tokenizer(formatted_prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode response (only the generated part)
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    
    return response.strip()

print("Ready for inference!")


In [ ]:
# Generate response using task_desc as system prompt and single_turn_prompt as user prompt
print("=" * 60)
print("GENERATING RESPONSE WITH SYSTEM + USER PROMPTS")
print("=" * 60)
print(f"System prompt: {task_desc}")
print(f"User prompt: {single_turn_prompt}")
print("\n" + "-" * 60)

# Create conversation with system and user messages
messages = [
    {"role": "system", "content": task_desc},
    {"role": "user", "content": single_turn_prompt}
]

# Apply chat template
formatted_prompt = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True
)

print("Formatted prompt:")
print(formatted_prompt)
print("\n" + "-" * 60)

# Tokenize
inputs = tokenizer(formatted_prompt, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

# Generate
print("Generating response...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.8,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

# Decode response (only the generated part)
generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("\nModel Response:")
print("=" * 60)
print(response.strip())
